# Lesson 3 exercise — Labels and masking

**Goal:** build labels that hide the prompt behind `-100`, so the loss counts only the answer. Then train briefly and see what changes: the model learns the answer and leaves the prompt alone.

**How to use this notebook**
1. In Colab: *Runtime → Change runtime type → T4 GPU*.
2. Run cells top to bottom. Wherever you see `...` after a `# TODO`, replace it with your code.
3. Run the **check** cell under each TODO. ✅ means you've got it; ❌ gives you a hint.

Stuck? Look up the error in the course's troubleshooting sheet (`reference/troubleshooting.html`). Still stuck? Ask on the Hugging Face Forums with your code and the full traceback.

## 0 · Setup
Installs the course's library versions (about a minute) and defines the `check` helper.

In [ ]:
%pip install -q -U "transformers>=5.19,<6" "trl>=1.15,<2" "peft>=0.21,<1" "datasets>=5,<6" "accelerate>=1.15,<2"
import torch, transformers
device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device:", device)

def check(passed, ok_msg, hint):
    print(("✅ " + ok_msg) if passed else ("❌ " + hint))

## 1 · Task, metric and model
The next two cells are shared by every lesson from here on: `make_rows()` generates the course's extraction task, and `score()` measures valid-JSON rate and field accuracy.

In [ ]:
import json, random

PROMPT = "Extract the person's name, age and city as JSON."
NAMES = ["Sara", "Omar", "Lina", "Youssef", "Mona", "Karim", "Nour", "Emma", "Liam", "Sofia",
         "Mateo", "Aiko", "Kenji", "Priya", "Arjun", "Chloe", "Lucas", "Fatima", "Diego", "Elena",
         "Leo", "Amara", "Kofi", "Hana", "Ines"]          # last 5 are test-only
CITIES = ["Cairo", "Alexandria", "Paris", "Lagos", "Tokyo", "Lima", "Berlin", "Toronto",
          "Mumbai", "Seoul", "Nairobi", "Madrid", "Dubai", "Oslo", "Austin", "Lisbon",
          "Hanoi", "Accra", "Quito", "Perth"]                # last 4 are test-only
TEMPLATES = [
    "Hi, I'm {name} from {city}. I'm {age}.",
    "{name} ({age}) lives in {city}.",
    "My name is {name}, I am {age} years old and I live in {city}.",
    "New signup: {name}, age {age}, based in {city}.",
    "Hello from {city}! {name} here, just turned {age}.",
    "{city}-based {name}, {age}, is joining the team next week.",
    "Contact: {name} | {age} | {city}",
    "Applicant {name} is {age} and currently resides in {city}.",
    "Meet {name}: {age} years young and loving life in {city}.",   # test-only
    "I moved to {city} last year. Name's {name}, age {age}.",      # test-only
]

def make_rows(n, split="train", seed=0):
    """Return n chat examples in TRL's prompt-completion format."""
    rng = random.Random(seed)
    test = split == "test"
    names = NAMES[20:] if test else NAMES[:20]
    cities = CITIES[16:] if test else CITIES[:16]
    templates = TEMPLATES[8:] if test else TEMPLATES[:8]
    rows = []
    for _ in range(n):
        person = {"name": rng.choice(names), "age": rng.randint(18, 80), "city": rng.choice(cities)}
        text = rng.choice(templates).format(**person)
        rows.append({
            "prompt": [{"role": "user", "content": f"{PROMPT}\n\nText: {text}"}],
            "completion": [{"role": "assistant", "content": json.dumps(person)}],
        })
    return rows

In [ ]:
def score(model, tokenizer, rows, show=3):
    """Greedy-generate a reply for each row; report valid-JSON rate and field accuracy."""
    model.eval()
    valid, fields_right = 0, 0
    for i, row in enumerate(rows):
        gold = json.loads(row["completion"][0]["content"])
        inputs = tokenizer.apply_chat_template(row["prompt"], add_generation_prompt=True,
                                               return_tensors="pt", return_dict=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
        reply = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            pred = json.loads(reply)
            valid += 1
        except json.JSONDecodeError:
            pred = {}
        if isinstance(pred, dict):
            fields_right += sum(pred.get(k) == v for k, v in gold.items())
        if i < show:
            print(f"  reply: {reply!r}")
    n = len(rows)
    result = {"valid_json": valid / n, "field_accuracy": fields_right / (3 * n)}
    print(result)
    return result

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).to(device)

rows = make_rows(8, "train", seed=0)       # 8 training examples
row = rows[0]
print("prompt:    ", row["prompt"][0]["content"])
print("completion:", row["completion"][0]["content"])

## 2 · Where does the answer start?
Render the whole conversation (prompt + completion) as in Lesson 2. Now render the prompt **alone**, with `add_generation_prompt=True`. It's exactly the beginning of the whole sequence, up to and including `<|im_start|>assistant`. So its length is the position where the answer starts.

**TODO 1:** In `prompt_length()`, render `row["prompt"]` alone, ending with the opening of the assistant's turn, as PyTorch tensors.

In [ ]:
def prompt_length(row):
    """Number of tokens before the answer starts."""
    # TODO 1
    prompt = ...
    return prompt["input_ids"].shape[1]

full = tok.apply_chat_template(row["prompt"] + row["completion"], return_tensors="pt", return_dict=True).to(device)
input_ids = full["input_ids"]
prompt_len = prompt_length(row)
print("whole sequence:", input_ids.shape[1], "tokens | prompt:", prompt_len, "tokens")

In [ ]:
# check 1
answer_part = tok.decode(input_ids[0, prompt_len:])
check(answer_part.startswith(row["completion"][0]["content"]),
      "Everything from position prompt_len onward is the answer.",
      "Render only row['prompt'] (no completion), and open the assistant's turn as you did at inference time in Lesson 1.")

In [ ]:
print(repr(answer_part))

The answer part ends with `<|im_end|>`. It stays in the loss, so the model also learns when to stop.

## 3 · Mask the prompt
Labels start as a copy of `input_ids`, as in Lesson 2. Then every prompt position becomes `-100`, the label value that PyTorch's cross-entropy skips by default.

**TODO 2:** Complete `masked_labels()`. Make a **copy** first: `labels = input_ids` would point at the same tensor, and writing `-100` into it would wreck the inputs too.

In [ ]:
def masked_labels(input_ids, prompt_len):
    """input_ids with every prompt position replaced by -100."""
    # TODO 2: copy input_ids, then set positions 0 .. prompt_len-1 to -100
    labels = ...
    ...
    return labels

labels = masked_labels(input_ids, prompt_len)
print(labels[0, prompt_len - 3 : prompt_len + 3])    # the boundary: -100s, then answer tokens

In [ ]:
# check 2
ok = (torch.is_tensor(labels) and labels.shape == input_ids.shape
      and bool((input_ids >= 0).all())                          # the inputs are untouched
      and bool((labels[0, :prompt_len] == -100).all())
      and torch.equal(labels[0, prompt_len:], input_ids[0, prompt_len:]))
check(ok, f"{prompt_len} prompt positions masked, {input_ids.shape[1] - prompt_len} answer tokens kept, inputs untouched.",
      "Copy first, then mask the first prompt_len positions along the sequence axis. "
      "If input_ids now contains -100, rerun the cell that builds `full`.")

## 4 · Same text, different labels
**TODO 3:** Compute the loss with your masked labels. The line above it, from Lesson 2, is your model.

In [ ]:
with torch.no_grad():
    loss_all = model(**full, labels=input_ids).loss       # Lesson 2: every token counts
    # TODO 3
    loss_answer = ...

n_all = (input_ids[:, 1:] != -100).sum().item()     # position 0 is never a target: nothing comes before it
n_answer = (labels[:, 1:] != -100).sum().item()
print(f"all tokens:   loss {loss_all.item():.3f} over {n_all} tokens")
print(f"answer only:  loss {loss_answer.item():.3f} over {n_answer} tokens")

In [ ]:
# check 3 — recompute it as plain cross-entropy; -100 is the default ignore_index
import torch.nn.functional as F
with torch.no_grad():
    logits = model(**full).logits
by_hand = F.cross_entropy(logits[0, :-1], labels[0, 1:])
ok = torch.is_tensor(loss_answer) and abs(loss_answer.item() - by_hand.item()) < 1e-4
check(ok, f"Matches cross-entropy by hand ({by_hand.item():.4f}): the -100 positions were skipped.",
      "Same model call as loss_all, but with the labels you built in TODO 2.")

Most of the tokens the unmasked loss counted were prompt: the system prompt and the user's message, text the user always supplies. The masked loss measures only what you want the model to learn: the answer.

## 5 · Train on answers only
This is the Lesson 2 loop, now over 8 rows with masked labels. Before and after, it measures `rows[0]`'s loss on its **prompt** tokens and on its **answer** tokens separately. If masking works, only the answer loss should move.

In [ ]:
examples = []
for r in rows:
    f = tok.apply_chat_template(r["prompt"] + r["completion"], return_tensors="pt", return_dict=True).to(device)
    examples.append((f, masked_labels(f["input_ids"], prompt_length(r))))

def prompt_vs_answer_loss():
    """rows[0]: loss on its prompt tokens only, and on its answer tokens only."""
    prompt_only = torch.where(labels == -100, input_ids, -100)    # your mask, flipped
    with torch.no_grad():
        return model(**full, labels=prompt_only).loss.item(), model(**full, labels=labels).loss.item()

print("before:  prompt loss %.3f | answer loss %.3f" % prompt_vs_answer_loss())

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
model.train()
for epoch in range(3):
    for f, lab in examples:
        out = model(**f, labels=lab)
        out.loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print(f"epoch {epoch}: last loss {out.loss.item():.4f}")
model.eval()

print("after:   prompt loss %.3f | answer loss %.3f" % prompt_vs_answer_loss())

The answer loss should collapse while the prompt loss barely moves: the model learned the answers and not the prompt. Now the real test, on 10 held-out rows with names, cities and phrasings it never saw:

In [ ]:
result = score(model, tok, make_rows(10, "test", seed=1))

Eight examples and 24 steps is a tiny run, so a small slip in one field is possible. Lesson 5 trains on hundreds of rows.

## Stretch (optional)
1. Rerun section 1 for a fresh model, then rerun sections 2–5 with `labels=f["input_ids"]` instead of `labels=lab` in the loop. What happens to the **prompt** loss now? What did the model spend its updates on?
2. Print `len(examples[0][1][0])` and `len(examples[1][1][0])`. To batch these two rows together you'd pad the shorter one. Which label value should the padding positions get?